# Curve-vs-Vol Basis — Grid Search

Sweeps the mean-reversion fade of the pair basis (curve vs option-implied median path)
across signal/exit/entry configs and reports which survive **honest execution**.

Honesty rules (non-negotiable, baked into `RVUtils.FlyVsVol.pair_backtest`):
- default marks = **listed wing premiums** at strikes fixed at entry (`basis='premium'`;
  `'skew'` = BL model marks, comparison only — model-marked results are an upper bound,
  never a tradeability verdict),
- **lag-1 execution**: entries/exits fill at the day *after* the signal bar,
- quality gates at entry (both legs |fwd residual| <= 2.5bp),
- flat round-trip cost per completed trade (4-leg RR-spread package, default 2.5bp).

Data: `notebooks/data/fly_vs_vol/{wing_premiums,contracts}.parquet`
(rebuild via `notebooks/rv/run_fly_vs_vol_screener.py` + the wing-panel builder).

In [1]:
import sys; sys.path.append("../../")

import dataclasses
import numpy as np
import pandas as pd
from pathlib import Path

from RVUtils.FlyVsVol.pair_backtest import (
    PairBacktestConfig, grid_search, prepare_data, run_pair_backtest,
)
from RVUtils.FlyVsVol.screener import series_half_life
from BT.signals.deflated_sharpe import deflated_sharpe

DATA = Path("../data/fly_vs_vol")
wings = pd.read_parquet(DATA / "wing_premiums.parquet")
contracts = pd.read_parquet(DATA / "contracts.parquet")
data = prepare_data(wings, contracts)
print(f"premium-basis rows: {len(data['premium'])}, pairs: {data['premium']['label'].nunique()}")

# ---- editable sweep ----------------------------------------------------------
BASE = PairBacktestConfig(round_trip_cost_bp=2.5, lag=1, quality_gate=True)
GRID = {
    "basis": ["premium", "skew"],
    "ma": [1, 5, 10],
    "zscore_window": [60, 120],
    "entry_min_zscore": [1.5, 2.0, 2.5],
    "exit_style": ["z0", "half", "t10"],
}
N_TRIALS = int(np.prod([len(v) for v in GRID.values()]))
print(f"grid size: {N_TRIALS} configs")

premium-basis rows: 2071, pairs: 7
grid size: 108 configs


In [2]:
# mean-reversion diagnostics of both basis definitions
for name, df_, col in (("premium-native", data["premium"], "prem_basis_bp"),
                       ("model (skew)", data["skew"], "pair_rent_skew_bp")):
    print(f"=== {name} basis ===")
    for label, sub in df_.groupby("label"):
        s = sub.sort_values("as_of")[col].reset_index(drop=True)
        d = s.diff().dropna()
        vr5 = s.diff(5).dropna().var() / (5 * d.var()) if d.var() > 0 else np.nan
        print(f"  {label}: n={len(s)} std={s.std():.2f}bp "
              f"hl_raw={series_half_life(s):.1f}d "
              f"hl_5dma={series_half_life(s.rolling(5).mean()):.1f}d "
              f"ac1(diff)={d.autocorr():+.2f} VR(5)={vr5:.2f}")

=== premium-native basis ===
  SFRH27-SFRM27: n=289 std=2.72bp hl_raw=infd hl_5dma=3.6d ac1(diff)=-0.54 VR(5)=0.18
  SFRH28-SFRM28: n=301 std=4.00bp hl_raw=0.4d hl_5dma=4.9d ac1(diff)=-0.46 VR(5)=0.23
  SFRM27-SFRU27: n=301 std=3.54bp hl_raw=0.3d hl_5dma=3.7d ac1(diff)=-0.46 VR(5)=0.21
  SFRU26-SFRZ26: n=289 std=2.47bp hl_raw=infd hl_5dma=3.3d ac1(diff)=-0.54 VR(5)=0.17
  SFRU27-SFRZ27: n=301 std=4.05bp hl_raw=0.4d hl_5dma=3.6d ac1(diff)=-0.35 VR(5)=0.24
  SFRZ26-SFRH27: n=289 std=1.91bp hl_raw=infd hl_5dma=3.7d ac1(diff)=-0.52 VR(5)=0.19
  SFRZ27-SFRH28: n=301 std=3.85bp hl_raw=0.3d hl_5dma=3.2d ac1(diff)=-0.45 VR(5)=0.22
=== model (skew) basis ===
  SFRH27-SFRM27: n=289 std=4.35bp hl_raw=0.9d hl_5dma=12.5d ac1(diff)=-0.44 VR(5)=0.27
  SFRH28-SFRM28: n=301 std=7.55bp hl_raw=1.2d hl_5dma=13.6d ac1(diff)=-0.31 VR(5)=0.32
  SFRM27-SFRU27: n=301 std=5.44bp hl_raw=0.7d hl_5dma=8.0d ac1(diff)=-0.35 VR(5)=0.26
  SFRU26-SFRZ26: n=289 std=3.71bp hl_raw=0.9d hl_5dma=8.0d ac1(diff)=-0.31 VR(5)=0

In [3]:
res = grid_search(GRID, data=data, base=BASE, show_progress=True)
res.to_csv("curve_vs_vol_grid_results.csv", index=False)
pd.set_option("display.width", 220)

print("=" * 100)
print(f"TOP 20 BY TOTAL NET (bp) — selection-inflated, see DSR cell")
print("=" * 100)
cols = ["config", "n_trades", "hit_rate", "avg_net_bp", "total_net_bp",
        "sharpe", "max_dd_bp", "avg_hold_days", "worst_bp"]
print(res.sort_values("total_net_bp", ascending=False).head(20)[cols]
      .round(2).to_string(index=False))
print()
print("TOP 10 BY SHARPE (n_trades >= 5):")
print(res[res["n_trades"] >= 5].sort_values("sharpe", ascending=False)
      .head(10)[cols].round(2).to_string(index=False))
print()
print("BOTTOM 5:")
print(res.sort_values("total_net_bp").head(5)[cols].round(2).to_string(index=False))

  20/108 configs


  40/108 configs


  60/108 configs


  80/108 configs


  100/108 configs


TOP 20 BY TOTAL NET (bp) — selection-inflated, see DSR cell
                       config  n_trades  hit_rate  avg_net_bp  total_net_bp  sharpe  max_dd_bp  avg_hold_days  worst_bp
 skew|ma10|w60|z2.0|half|lag1         5      1.00        3.32         16.59    1.78      -8.05          12.20      0.02
  skew|ma10|w60|z2.0|t10|lag1         5      1.00        2.89         14.47    1.87      -9.51          10.00      1.28
skew|ma10|w120|z1.5|half|lag1         8      0.62        1.52         12.20    0.61     -12.38          10.25     -1.65
   skew|ma10|w60|z2.0|z0|lag1         5      0.80        2.04         10.21    0.73     -10.04          15.20     -2.94
  skew|ma10|w120|z1.5|z0|lag1         8      0.50        1.05          8.43    0.29     -17.81          13.88     -3.58
   skew|ma5|w120|z2.0|z0|lag1         4      0.50        1.00          4.02    0.37     -12.92          10.50     -1.23
  skew|ma10|w120|z2.0|z0|lag1         4      0.25        0.94          3.76    0.35     -10.04      

In [4]:
print("=" * 100)
print("PARAMETER SENSITIVITY (median across configs sharing the value)")
print("=" * 100)
for k in GRID:
    med = res.groupby(k)[["total_net_bp", "sharpe", "n_trades"]].median().round(2)
    print(f"\n{k}:")
    print(med.to_string())

print()
print("=" * 100)
print("DISTRIBUTION ACROSS THE GRID (overfitting check)")
print("=" * 100)
print(f"  configs net-positive:        {(res['total_net_bp'] > 0).mean():.0%}")
print(f"  median total_net:            {res['total_net_bp'].median():+.2f}bp")
print(f"  median sharpe:               {res['sharpe'].median():+.2f}")
print(f"  configs sharpe > 0 / 0.5 / 1: "
      f"{(res['sharpe'] > 0).sum()} / {(res['sharpe'] > 0.5).sum()} / "
      f"{(res['sharpe'] > 1.0).sum()}  of {len(res)}")
print(f"  premium-basis configs net-positive: "
      f"{(res[res['basis'] == 'premium']['total_net_bp'] > 0).mean():.0%}")

PARAMETER SENSITIVITY (median across configs sharing the value)

basis:
         total_net_bp  sharpe  n_trades
basis                                  
premium        -13.25   -2.77       6.0
skew            -1.31   -0.19       4.0

ma:
    total_net_bp  sharpe  n_trades
ma                                
1          -4.63   -1.33       2.0
5          -4.42   -0.83       5.0
10         -4.72   -1.05       5.5

zscore_window:
               total_net_bp  sharpe  n_trades
zscore_window                                
60                    -6.62   -1.13       4.5
120                   -2.37   -1.02       4.0

entry_min_zscore:
                  total_net_bp  sharpe  n_trades
entry_min_zscore                                
1.5                     -17.88   -1.49      10.0
2.0                      -2.76   -1.06       4.0
2.5                      -2.09   -0.89       1.0

exit_style:
            total_net_bp  sharpe  n_trades
exit_style                                
half               -4.71 

In [5]:
# best PREMIUM-marked config: deflated Sharpe + neighborhood stability + lag ablation
prem_res = res[(res["basis"] == "premium") & (res["n_trades"] >= 5)]
best = prem_res.sort_values("total_net_bp", ascending=False).iloc[0]
best_cfg = dataclasses.replace(
    BASE, basis="premium", ma=int(best["ma"]), zscore_window=int(best["zscore_window"]),
    entry_min_zscore=float(best["entry_min_zscore"]), exit_style=best["exit_style"])
bt = run_pair_backtest(best_cfg, data=data)
print(f"best premium config: {best_cfg.label()}")
print({k: round(v, 2) if isinstance(v, float) else v for k, v in bt.metrics.items()})

if len(bt.daily_pnl) > 20:
    dsr = deflated_sharpe(bt.daily_pnl.values, n_trials=N_TRIALS)
    print("\ndeflated Sharpe vs", N_TRIALS, "trials:")
    print({k: (round(v, 3) if isinstance(v, (int, float)) else v) for k, v in dsr.items()})

print("\nneighborhood stability (one param moved at a time):")
for k, vals in GRID.items():
    if k == "basis":
        continue
    for v in vals:
        if v == getattr(best_cfg, k):
            continue
        nb = run_pair_backtest(dataclasses.replace(best_cfg, **{k: v}), data=data)
        print(f"  {k}={v}: total_net {nb.metrics['total_net_bp']:+.1f}bp "
              f"(n={nb.metrics['n_trades']})")

lag0 = run_pair_backtest(dataclasses.replace(best_cfg, lag=0), data=data)
print(f"\nsame-bar (lag=0) inflation check: lag0 {lag0.metrics['total_net_bp']:+.1f}bp "
      f"vs lag1 {bt.metrics['total_net_bp']:+.1f}bp")

best premium config: premium|ma5|w60|z2.0|z0|lag1
{'n_trades': 7, 'total_net_bp': -6.5, 'hit_rate': 0.29, 'avg_net_bp': -0.93, 'sharpe': -1.06, 'max_dd_bp': -10.75, 'avg_hold_days': 9.29, 'worst_bp': -7.75}

deflated Sharpe vs 108 trials:
{'sr': -0.067, 'sr_annualised': -1.057, 'sr0': 0.0, 'sigma_sr': 0.124, 'dsr_z': -0.538, 'dsr_prob': 0.295, 'skew': -0.319, 'kurt': 3.15, 'T': 65}

neighborhood stability (one param moved at a time):
  ma=1: total_net -17.0bp (n=3)
  ma=10: total_net -15.2bp (n=6)
  zscore_window=120: total_net -13.0bp (n=6)


  entry_min_zscore=1.5: total_net -25.2bp (n=15)
  entry_min_zscore=2.5: total_net -1.0bp (n=2)
  exit_style=half: total_net -13.5bp (n=8)
  exit_style=t10: total_net -19.8bp (n=8)



same-bar (lag=0) inflation check: lag0 -10.5bp vs lag1 -6.5bp


## Reading the results

- The **distribution block** is the verdict, not the top row: with ~100 configs, the best
  cell is selection-inflated by construction (that is what the deflated-Sharpe cell
  quantifies). A strategy is real when the *median* config survives costs and the best
  config's neighborhood is flat, not spiky.
- `skew`-basis rows mark P&L on BL model values — upper bound only. The premium rows are
  the executable ones.
- Findings on 2025-06..2026-07 (see spec + `curve_vs_vol_basis_backtest.py`): the basis
  mean-reverts strongly (VR(5) ~ 0.3) and the fade is gross-positive, but per-trade gross
  (~0.5-1.7bp) is below the 4-leg round-trip cost — viable only with cheaper structures,
  event-window entries, or maker execution.